In [ ]:
# Install Dependencies

!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.7 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1

In [ ]:
# Download Dataset 

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=api_key)
project = rf.workspace("michael-8jeqe").project("hardhat-detection-iukt9")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Hardhat-Detection-1 in yolov8:: 100%|██████████| 61918/61918 [00:06<00:00, 9041.86it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Validate FP32 Baseline Model

from ultralytics import YOLO
model = YOLO('/kaggle/input/models/cherianbiju/best-weight/pytorch/default/1/best.pt')
metrics = model.val(data=f'{dataset.location}/data.yaml')
print("FP32 mAP50:", metrics.box.map50)
print("FP32 mAP50-95:", metrics.box.map)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1581.9±564.4 MB/s, size: 60.3 KB)
val: Scanning /kaggle/working/Hardhat-Detection-1/valid/labels... 4281 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4281/4281 1.3Kit/s 3.2s<0.0s
val: New cache created: /kaggle/working/Hardhat-Detection-1/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 268/268 4.8it/s 56.2s<0.2s
                   all       4281      32532      0.682      0.562      0.573      0.289
                Helmet       3718      10433      0.743      0.716      0.665      0.334
             No-Helmet       1883      15418      0.793      0.513      0.582      0.266
                Person       1823       6681      0.509      0.458      0.471      0.267
Speed: 0.8ms preprocess, 8.7m

In [ ]:
# Validate FP16 ONNX Quantized Model

!yolo task=detect mode=val model='/kaggle/input/models/cherianbiju/best-weight-onnx/onnx/default/1/best.onnx' data={dataset.location}/data.yaml half=True

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /kaggle/input/models/cherianbiju/best-weight-onnx/onnx/default/1/best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 325ms
Prepared 1 package in 2.88s
Installed 1 package in 10ms
 + onnxruntime-gpu==1.26.0

requirements: AutoUpdate success ✅ 3.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Using ONNX Runtime 1.26.0 with CUDAExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1613.1±625.4 MB/s, size: 66.7 KB)
val: Scanning /kaggle/working/Hardhat-Detection-1/valid/labels.cache... 4281 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4281/4281 855.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):

In [ ]:
# to download the results as zip

import shutil
shutil.make_archive('/kaggle/working/runs', 'zip', '/kaggle/working/runs')
print('Done!')

Done!
